# Class-balanced T1 training on a MacBook Pro (MLX)

Local Apple Silicon version of `colab_t1_balanced_train.ipynb`. Run from this repository in an arm64 Python Jupyter kernel. The notebook reads the existing prepared images and mixed instruction JSONL in `data/secondary/` (no Colab upload or tar extraction). It trains **only T1** and evaluates **only `val.jsonl`**.

**Setup:** In a terminal, create/select a separate Apple Silicon Python 3.12+ environment with Jupyter, open this notebook from the repository directory, and accept the terms at [google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it) with your Hugging Face account. Have a token with read access available as `HF_TOKEN` (or enter it when prompted). The full prepared microscopy folder (~2.7 GB) must be present. Keep ample free disk space for the gated BF16 download, the separate 4-bit conversion, and the adapter; conversion may temporarily need both model copies. With 24 GB unified memory, image training may still run out of memory; complete the four-record smoke pass first.

The [MLX-VLM LoRA guide](https://github.com/Blaizzy/mlx-vlm/blob/v0.7.3/mlx_vlm/LORA.MD), [trainer source](https://github.com/Blaizzy/mlx-vlm/blob/v0.7.3/mlx_vlm/trainer/sft_trainer.py), and [conversion CLI](https://github.com/Blaizzy/mlx-vlm/blob/v0.7.3/mlx_vlm/convert.py) document the multimodal dataset, QLoRA, generation and revision-pinned conversion used below. Google's [MedGemma model card](https://huggingface.co/google/medgemma-4b-it) documents the July 2025 image-token fix. The old `mlx-community/medgemma-4b-it-4bit` was last updated before that fix, so convert the repository's **pinned** source revision here rather than using it. MLX 4-bit affine quantization and completion-only loss differ from Colab's CUDA NF4/FP32 setup; results and MLX adapter files are not bitwise or PEFT-compatible.

In [ ]:
%pip install -q 'mlx-vlm[train]==0.7.3' 'scikit-learn>=1.5'
import platform
import mlx.core as mx
import importlib.metadata as metadata
print('CPU:', platform.machine(), '| MLX:', metadata.version('mlx'), '| mlx-vlm:', metadata.version('mlx-vlm'))
if platform.machine() != 'arm64' or not mx.metal.is_available():
    raise RuntimeError('Select a native Apple Silicon Python kernel with working Metal support')

In [ ]:
from pathlib import Path
import subprocess, sys, json, os, getpass, shutil

# If Jupyter was started elsewhere, set ROOT explicitly to the repository path.
ROOT = Path.cwd().resolve()
TRAINER = ROOT / 'scripts/train_t1_mlx.py'
MODEL_DIR = ROOT / 'checkpoints/mlx-medgemma-4b-it-4bit'
OUTPUT_DIR = ROOT / 'adapters/medgemma-t1-mlx-balanced'
SMOKE_DIR = ROOT / 'adapters/medgemma-t1-mlx-smoke'
assert TRAINER.is_file(), f'Set ROOT to the repository directory: {ROOT}'
assert (ROOT / 'configs/secondary.json').is_file()
print('Root:', ROOT, '| Free disk (GiB):', round(shutil.disk_usage(ROOT).free / 2**30, 1))
subprocess.run([sys.executable, '-u', str(TRAINER), '--root', str(ROOT), '--mode', 'check'], check=True)

## Convert the pinned multimodal checkpoint
This downloads the gated BF16 weights (unless already cached) and writes an MLX 4-bit **language-decoder** quantization with the vision tower kept at its native precision. The local `model_source.json` prevents inadvertently training with a different revision. Allow substantial free disk space before proceeding; the source weights are cached separately. If conversion is interrupted, remove the **incomplete conversion directory only** and rerun.

In [ ]:
MODEL_ID = 'google/medgemma-4b-it'
REVISION = '290cda5eeccbee130f987c4ad74a59ae6f196408'
SOURCE = {'model_id': MODEL_ID, 'revision': REVISION, 'quantization': 'mlx-4bit-affine'}
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token for accepted MedGemma terms: ')
if not os.environ['HF_TOKEN']:
    raise RuntimeError('MedGemma access requires an HF_TOKEN')
marker = MODEL_DIR / 'model_source.json'
if MODEL_DIR.exists():
    if not marker.is_file() or json.loads(marker.read_text()) != SOURCE:
        raise RuntimeError(f'Existing conversion is incomplete or mismatched: {MODEL_DIR}')
    print('Reusing pinned conversion:', MODEL_DIR)
else:
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, '-m', 'mlx_vlm', 'convert',
        '--hf-path', MODEL_ID, '--revision', REVISION, '--mlx-path', str(MODEL_DIR),
        '--quantize', '--q-bits', '4', '--q-group-size', '64'], check=True)
    assert (MODEL_DIR / 'config.json').is_file() and (MODEL_DIR / 'tokenizer_config.json').is_file()
    marker.write_text(json.dumps(SOURCE, indent=2) + '\n')
print('Conversion:', MODEL_DIR)

## Train: four-record smoke, then one full balanced pass
The sampler matches the Colab class balancing (2,166 positive + 65 negative source records → 4,332 interleaved examples); batch size 1, seed 42, rank 8, alpha 16, dropout 0.05, LR 2e-4, accumulation 8. The four-record smoke and full run have **separate output directories**. The trainer flushes the final partial gradient group, giving 542 full-run optimizer updates. MLX-VLM's completion mask trains on the assistant classification only, keeping the image in the input. Do not use the smoke adapter for validation.

In [ ]:
def train_command(output, extra=()):
    return [sys.executable, '-u', str(TRAINER), '--root', str(ROOT),
        '--model-dir', str(MODEL_DIR), '--output', str(output), '--mode', 'train',
        '--epochs', '1', '--learning-rate', '2e-4',
        '--gradient-accumulation-steps', '8', '--seed', '42', *extra]

if (SMOKE_DIR / 'adapters.safetensors').is_file():
    print('Smoke adapter already exists:', SMOKE_DIR)
else:
    print('Running four-record smoke test...', flush=True)
    subprocess.run(train_command(SMOKE_DIR, ['--max-train', '4']), check=True)
assert (SMOKE_DIR / 'adapter_config.json').is_file()
assert (SMOKE_DIR / 'adapters.safetensors').is_file()
print('Smoke passed')

In [ ]:
if (OUTPUT_DIR / 'adapters.safetensors').is_file():
    print('Full-run adapter already exists:', OUTPUT_DIR)
else:
    print('Starting all 4,332 balanced examples...', flush=True)
    subprocess.run(train_command(OUTPUT_DIR), check=True)
assert (OUTPUT_DIR / 'adapter_config.json').is_file()
summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text())
assert summary['train_records'] == 4332 and summary['optimizer_updates'] == 542, summary
print(json.dumps(summary, indent=2))

## Validation and archive
The MLX adapter is loaded against the **same local quantized base** for greedy 8-token classification of the 488 validation examples. Predictions, confusion matrix, metrics and training summary are kept separately from Colab/PEFT artifacts. The protocol's selection gate is positive negative-control recall and macro-F1 above 0.4917. No test split is loaded.

In [ ]:
subprocess.run([sys.executable, '-u', str(TRAINER), '--root', str(ROOT),
    '--model-dir', str(MODEL_DIR), '--output', str(OUTPUT_DIR), '--mode', 'eval',
    '--max-val', '488', '--max-new-tokens', '8'], check=True)
metrics = json.loads((OUTPUT_DIR / 't1_metrics.json').read_text())
assert metrics['records'] == 488, metrics
print('Invalid outputs:', metrics['invalid_outputs'])
print('Selection gate:', metrics['negative_control_recall'] > 0 and metrics['macro_f1'] > 0.4917)
print('MLX adapter directory:', OUTPUT_DIR)

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print('Archived adapter, training summary, validation predictions and metrics:', archive)
print('Keep MODEL_DIR to run inference: this ZIP contains an MLX adapter, not the base model or a PEFT adapter.')

## Optional: export a Transformers/PEFT adapter
Use this if you need `adapter_model.safetensors` and standard PEFT `adapter_config.json` for Transformers/PEFT workflows. The exporter transposes and renames the MLX LoRA tensors. It does **not** retrain or convert the base model: this adapter was learned against the pinned MedGemma revision converted to MLX affine 4-bit (group size 64). Loading it against the original unquantized HF model is supported format-wise, but can produce different predictions from the MLX-trained model. The original MLX artifact remains available in `OUTPUT_DIR`.

In [ ]:
PEFT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + '-peft')
if not (PEFT_DIR / 'adapter_model.safetensors').is_file():
    subprocess.run([sys.executable, str(ROOT / 'scripts/export_mlx_adapter_peft.py'),
        '--input', str(OUTPUT_DIR), '--output', str(PEFT_DIR)], check=True)
assert (PEFT_DIR / 'adapter_config.json').is_file()
assert (PEFT_DIR / 'adapter_model.safetensors').is_file()
peft_zip = shutil.make_archive(str(PEFT_DIR), 'zip', root_dir=PEFT_DIR.parent, base_dir=PEFT_DIR.name)
print('PEFT adapter ZIP:', peft_zip)
print('PEFT directory:', PEFT_DIR)